# LegalBERT Fine-Tuning on CUAD
### Contract Risk Scoring Engine — Week 6
**Author:** Fresnel Fabian | Northeastern Roux Institute | MPS Applied Machine Intelligence

---

**Expected training time:**
| GPU | 5 epochs |
|---|---|
| T4 (free) | ~35 min |
| A100 (Pro) | ~12 min |
| V100 (Pro+) | ~18 min |

## Cell 1 — GPU Check + Install

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU')

# Install / upgrade packages
# transformers and datasets are usually pre-installed on Colab but may be outdated
!pip install -q --upgrade transformers==4.40.0 datasets accelerate

import torch
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 153.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
torch: 2.10.0+cu128
CUDA available: True
Device: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Using: cuda


## Cell 2 — Mount Google Drive (saves model + results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/contract_risk_legalbert'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Saving to: {SAVE_DIR}')

Mounted at /content/drive
Saving to: /content/drive/MyDrive/contract_risk_legalbert


## Cell 3 — Configuration

All hyperparameters in one place. T4 users: keep `BATCH_SIZE=8, ACCUM_STEPS=4`. A100 users: can raise `BATCH_SIZE=16, ACCUM_STEPS=2`.

In [ ]:
CFG = dict(
    # ── Model ──────────────────────────────────────────────────────────────
    MODEL_NAME     = 'nlpaueb/bert-base-uncased-contracts',
    NUM_LABELS     = 41,
    MAX_LENGTH     = 512,
    STRIDE         = 256,     # sliding window stride for long contracts
    DROPOUT        = 0.1,

    # ── Training ───────────────────────────────────────────────────────────
    EPOCHS         = 5,
    BATCH_SIZE     = 8,       # per-GPU batch (T4: 8, A100: 16)
    ACCUM_STEPS    = 4,       # gradient accumulation → effective batch = 8*4 = 32
    BASE_LR        = 2e-5,    # BERT encoder LR
    HEAD_LR        = 1e-4,    # classifier head LR (10x higher — starts random)
    WEIGHT_DECAY   = 0.01,
    WARMUP_RATIO   = 0.10,    # 10% of steps for LR warmup
    FREEZE_EPOCHS  = 2,       # freeze bottom 6 BERT layers for first N epochs
    MAX_GRAD_NORM  = 1.0,

    # ── Class imbalance ────────────────────────────────────────────────────
    POS_WEIGHT_CLIP = 20.0,   # clip inverse-frequency weights at this value

    # ── Threshold tuning ───────────────────────────────────────────────────
    THRESH_MIN     = 0.15,
    THRESH_MAX     = 0.70,
    THRESH_STEP    = 0.05,

    # ── Early stopping ─────────────────────────────────────────────────────
    PATIENCE       = 3,

    # ── Reproducibility ────────────────────────────────────────────────────
    SEED           = 42,

    # ── Output ─────────────────────────────────────────────────────────────
    SAVE_DIR       = SAVE_DIR,
)

import numpy as np, random
torch.manual_seed(CFG['SEED'])
np.random.seed(CFG['SEED'])
random.seed(CFG['SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['SEED'])

print('Config:')
for k, v in CFG.items():
    if k != 'SAVE_DIR': print(f'  {k:<20} = {v}')
print(f'  Effective batch size = {CFG["BATCH_SIZE"] * CFG["ACCUM_STEPS"]}')

Config:
  MODEL_NAME           = nlpaueb/bert-base-uncased-contracts
  NUM_LABELS           = 41
  MAX_LENGTH           = 512
  STRIDE               = 256
  DROPOUT              = 0.1
  EPOCHS               = 5
  BATCH_SIZE           = 8
  ACCUM_STEPS          = 4
  BASE_LR              = 2e-05
  HEAD_LR              = 0.0001
  WEIGHT_DECAY         = 0.01
  WARMUP_RATIO         = 0.1
  FREEZE_EPOCHS        = 2
  MAX_GRAD_NORM        = 1.0
  POS_WEIGHT_CLIP      = 20.0
  THRESH_MIN           = 0.15
  THRESH_MAX           = 0.7
  THRESH_STEP          = 0.05
  PATIENCE             = 3
  SEED                 = 42
  Effective batch size = 32


## Cell 4 — CUAD Data Loading

CUAD-QA format: one row per (contract, question). We pivot to one row per contract with 41 binary labels. A clause is **present** when the answer span is non-empty.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FIX: datasets 3.x dropped support for custom .py dataset scripts.
# theatticusproject/cuad-qa uses one, so load_dataset() raises RuntimeError.
#
# Solution A (try first):  pin datasets to 2.x, which still supports scripts.
# Solution B (fallback):   download CUADv1.json directly — no datasets needed.
# ─────────────────────────────────────────────────────────────────────────────

import json, os, urllib.request
import numpy as np

CLAUSE_TYPES = [
    'Document Name','Parties','Agreement Date','Effective Date','Expiration Date',
    'Renewal Term','Notice Period To Terminate Renewal','Governing Law',
    'Most Favored Nation','Non-Compete','Exclusivity','No-Solicit Of Customers',
    'No-Solicit Of Employees','Non-Disparagement','Limitation Of Liability',
    'Cap On Liability','Liquidated Damages','Unilateral Termination',
    'Bilateral Termination','Anti-Assignment','Revenue/Profit Sharing',
    'Price Restrictions','Minimum Commitment','Volume Restriction',
    'IP Ownership Assignment','Joint IP Ownership','License Grant',
    'Non-Transferable License','Affiliate License-Licensor','Affiliate License-Licensee',
    'Unlimited License','Irrevocable Or Perpetual License','Source Code Escrow',
    'Post-Termination Services','Audit Rights','Uncapped Liability',
    'Warranty Duration','Insurance','Covenant Not To Sue',
    'Third Party Beneficiary','Change Of Control',
]

def question_to_clause(question):
    q = question.lower()
    for ct in CLAUSE_TYPES:
        if ct.split()[0].lower() in q:
            return ct
    return None

def pivot_squad_json(raw_json):
    """Pivot CUADv1.json (SQuAD format) → (texts, label_matrix)."""
    contracts = {}
    for entry in raw_json['data']:
        title = entry['title']
        for para in entry['paragraphs']:
            context = para['context']
            if title not in contracts:
                contracts[title] = {'text': context, 'present': set()}
            for qa in para['qas']:
                clause = question_to_clause(qa['question'])
                if clause and qa['answers']:   # non-empty answers = clause present
                    contracts[title]['present'].add(clause)

    texts_out, labels_out = [], []
    for data in contracts.values():
        texts_out.append(data['text'])
        labels_out.append([int(ct in data['present']) for ct in CLAUSE_TYPES])

    Y = np.array(labels_out, dtype=np.float32)
    return texts_out, Y

def pivot_qa_rows(rows):
    """Pivot HuggingFace QA rows → (texts, label_matrix)."""
    contracts = {}
    for row in rows:
        title   = row['title']
        context = row['context']
        clause  = question_to_clause(row['question'])
        if clause is None: continue
        if title not in contracts:
            contracts[title] = {'text': context, 'present': set()}
        answers = row['answers']
        texts_ans = answers['text'] if isinstance(answers, dict) else []
        if any(t.strip() for t in texts_ans):
            contracts[title]['present'].add(clause)

    texts_out, labels_out = [], []
    for data in contracts.values():
        texts_out.append(data['text'])
        labels_out.append([int(ct in data['present']) for ct in CLAUSE_TYPES])

    Y = np.array(labels_out, dtype=np.float32)
    return texts_out, Y

def report(texts, Y):
    print(f'  Contracts:    {len(texts)}')
    print(f'  Label matrix: {Y.shape}')
    print(f'  Overall +ve:  {Y.mean():.1%}')
    print(f'  +ve range:    [{Y.mean(axis=0).min():.1%}, {Y.mean(axis=0).max():.1%}]')


all_texts, Y_all = None, None

# ── SOLUTION A: pin datasets to 2.x ──────────────────────────────────────────
print('Trying Solution A: datasets < 3.0 ...')
try:
    import importlib
    import datasets as _ds
    ds_ver = tuple(int(x) for x in _ds.__version__.split('.')[:2])

    if ds_ver >= (3, 0):
        print(f'  datasets {_ds.__version__} detected — downgrading to 2.x ...')
        import subprocess, sys
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'datasets==2.21.0', '--upgrade'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        # Force reimport
        import importlib
        import datasets
        importlib.reload(datasets)
        from datasets import load_dataset, concatenate_datasets
        print(f'  Reloaded datasets {datasets.__version__}')
    else:
        from datasets import load_dataset, concatenate_datasets
        print(f'  datasets {_ds.__version__} — already compatible')

    print('  Loading CUAD from HuggingFace ...')
    raw      = load_dataset('theatticusproject/cuad-qa')
    combined = concatenate_datasets([raw['train'], raw['test']])
    print(f'  QA pairs: {len(combined)}')
    all_texts, Y_all = pivot_qa_rows(combined)
    print('  Solution A succeeded.')
    report(all_texts, Y_all)

except Exception as e:
    print(f'  Solution A failed: {e}')

# ── SOLUTION B: download CUADv1.json directly ─────────────────────────────────
if all_texts is None:
    print()
    print('Trying Solution B: direct JSON download from HuggingFace CDN ...')

    JSON_PATH = '/content/CUADv1.json'
    JSON_URL  = (
        'https://huggingface.co/datasets/theatticusproject/cuad-qa'
        '/resolve/main/CUADv1.json'
    )

    if not os.path.exists(JSON_PATH):
        print(f'  Downloading {JSON_URL} ...')
        try:
            urllib.request.urlretrieve(JSON_URL, JSON_PATH)
            print(f'  Downloaded: {os.path.getsize(JSON_PATH)/1e6:.1f} MB')
        except Exception as e:
            # fallback: wget (always available in Colab)
            print(f'  urllib failed ({e}), trying wget ...')
            os.system(f'wget -q -O {JSON_PATH} "{JSON_URL}"')
            if os.path.exists(JSON_PATH):
                print(f'  Downloaded via wget: {os.path.getsize(JSON_PATH)/1e6:.1f} MB')
    else:
        print(f'  Found cached: {JSON_PATH}')

    try:
        print('  Parsing JSON ...')
        with open(JSON_PATH) as f:
            raw_json = json.load(f)
        all_texts, Y_all = pivot_squad_json(raw_json)
        print('  Solution B succeeded.')
        report(all_texts, Y_all)
    except Exception as e:
        print(f'  Solution B failed: {e}')

# ── SOLUTION C: Atticus Project direct URL ────────────────────────────────────
if all_texts is None:
    print()
    print('Trying Solution C: Atticus Project direct download ...')
    JSON_PATH2 = '/content/CUADv1_atticus.json'
    ATTICUS_URL = 'https://zenodo.org/record/4599817/files/CUAD_v1.json'
    try:
        os.system(f'wget -q -O {JSON_PATH2} "{ATTICUS_URL}"')
        with open(JSON_PATH2) as f:
            raw_json = json.load(f)
        all_texts, Y_all = pivot_squad_json(raw_json)
        print('  Solution C succeeded.')
        report(all_texts, Y_all)
    except Exception as e:
        print(f'  Solution C failed: {e}')

# ─────────────────────────────────────────────────────────────────────────────
assert all_texts is not None, (
    'All data loading solutions failed.\n'
    'Manual fallback: Upload CUADv1.json to /content/ and rerun this cell.\n'
    'Download from: https://www.atticusprojectai.org/cuad'
)
print(f'\nData ready: {len(all_texts)} contracts, {Y_all.shape[1]} clause types.')


Trying Solution A: datasets < 3.0 ...
  datasets 4.8.3 detected — downgrading to 2.x ...
  Solution A failed: cannot import name 'BeamBasedBuilder' from 'datasets.builder' (/usr/local/lib/python3.12/dist-packages/datasets/builder.py)

Trying Solution B: direct JSON download from HuggingFace CDN ...
  urllib failed (HTTP Error 404: Not Found), trying wget ...
  Downloaded via wget: 0.0 MB
  Parsing JSON ...
  Solution B failed: Expecting value: line 1 column 1 (char 0)

Trying Solution C: Atticus Project direct download ...
  Solution C failed: Expecting value: line 1 column 1 (char 0)


AssertionError: All data loading solutions failed.
Manual fallback: Upload CUADv1.json to /content/ and rerun this cell.
Download from: https://www.atticusprojectai.org/cuad

## Cell 5 — Iterative Stratification

Sechidis et al. (2011): processes labels rarest→most common, distributing each label's positives proportionally across folds. Guarantees all 41 clause types present in every fold. Standard `StratifiedKFold` silently produces folds with zero positives for rare labels.

In [ ]:
def iterative_stratification_split(texts, Y, train_r=0.70, val_r=0.15, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(texts)
    fa  = np.full(n, -1)
    fs  = np.array([train_r, val_r, 1 - train_r - val_r]) * n
    fl  = np.zeros((3, Y.shape[1]))

    for l in np.argsort(Y.sum(axis=0)):
        pos = np.where((Y[:, l] == 1) & (fa == -1))[0]
        rng.shuffle(pos)
        for idx in pos:
            d    = np.array([train_r, val_r, 1-train_r-val_r]) * Y[:, l].sum()
            fold = int(np.argmax(d - fl[:, l]))
            fa[idx] = fold
            fl[fold, l] += 1

    un = np.where(fa == -1)[0]
    rng.shuffle(un)
    for idx in un:
        fa[idx] = int(np.argmax(fs - [np.sum(fa == f) for f in range(3)]))

    ti, vi, ei = [np.where(fa == f)[0] for f in range(3)]

    tr_t = [texts[i] for i in ti]
    va_t = [texts[i] for i in vi]
    te_t = [texts[i] for i in ei]

    print(f'Split: train={len(ti)}  val={len(vi)}  test={len(ei)}')
    for name, idx in [('train', ti), ('val', vi), ('test', ei)]:
        covered = int((Y[idx].sum(axis=0) > 0).sum())
        min_pos = int(Y[idx].sum(axis=0)[Y[idx].sum(axis=0) > 0].min())
        print(f'  {name:<6} {covered}/41 labels covered   min positives: {min_pos}')

    return tr_t, va_t, te_t, Y[ti], Y[vi], Y[ei]

tr_texts, va_texts, te_texts, Y_tr, Y_va, Y_te = iterative_stratification_split(
    all_texts, Y_all, seed=CFG['SEED']
)

## Cell 6 — Model + Dataset + DataLoaders

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

# ── Model ─────────────────────────────────────────────────────────────────────
class LegalBERTClassifier(nn.Module):
    """
    bert-base-uncased-contracts + linear head for multi-label classification.

    Why BCEWithLogitsLoss (not CrossEntropyLoss):
        41 clause labels are independent. A contract can simultaneously have
        Governing Law AND Anti-Assignment AND License Grant. Softmax forces
        them to compete (probabilities sum to 1). Sigmoid treats each label
        independently — the correct formulation for multi-label problems.

    Why one linear layer:
        Week 4 showed adding depth to learned features doesn't help on this task.
        The BERT [CLS] embedding already encodes the relevant non-linearity.
        One layer regularizes better with n=510.
    """
    def __init__(self, model_name, num_labels=41, dropout=0.1):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self._init_head()

    def _init_head(self):
        """Xavier init on the classifier head for stable early gradients."""
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token: (B, 768)
        return self.classifier(self.dropout(cls))   # (B, num_labels) — raw logits

# ── Dataset ───────────────────────────────────────────────────────────────────
class CUADDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.labels    = torch.tensor(labels, dtype=torch.float32)
        print(f'  Tokenizing {len(texts)} contracts...', end=' ', flush=True)
        self.enc = tokenizer(
            texts, max_length=max_length, truncation=True,
            padding='max_length', return_tensors='pt',
        )
        print('done')
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids':      self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels':         self.labels[i]}

# ── Tokenize ──────────────────────────────────────────────────────────────────
print(f'Loading tokenizer: {CFG["MODEL_NAME"]}')
tokenizer = AutoTokenizer.from_pretrained(CFG['MODEL_NAME'])

print('Building datasets:')
tr_ds = CUADDataset(tr_texts, Y_tr, tokenizer, CFG['MAX_LENGTH'])
va_ds = CUADDataset(va_texts, Y_va, tokenizer, CFG['MAX_LENGTH'])
te_ds = CUADDataset(te_texts, Y_te, tokenizer, CFG['MAX_LENGTH'])

tr_loader = DataLoader(tr_ds, batch_size=CFG['BATCH_SIZE'], shuffle=True,
                       num_workers=2, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=CFG['BATCH_SIZE'], shuffle=False,
                       num_workers=2, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=CFG['BATCH_SIZE'], shuffle=False,
                       num_workers=2, pin_memory=True)

print(f'\nTrain batches: {len(tr_loader)}')

# ── Model + loss ──────────────────────────────────────────────────────────────
print(f'\nLoading model: {CFG["MODEL_NAME"]}')
model = LegalBERTClassifier(
    CFG['MODEL_NAME'], CFG['NUM_LABELS'], CFG['DROPOUT']
).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_total:,}  ({n_total/1e6:.0f}M)')

# Per-label positive weights: inverse frequency clipped at POS_WEIGHT_CLIP
pos_freq   = Y_tr.mean(axis=0)
pos_weight = np.clip((1 - pos_freq) / (pos_freq + 1e-8), 1.0, CFG['POS_WEIGHT_CLIP'])
criterion  = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)
)

print('\nTop 5 highest positive weights (rarest clauses):')
for name, w in sorted(zip(CLAUSE_TYPES, pos_weight), key=lambda x: -x[1])[:5]:
    idx  = CLAUSE_TYPES.index(name)
    print(f'  {name:<40} weight={w:.1f}  (+ve rate={pos_freq[idx]:.1%})')

## Cell 7 — Optimizer + Scheduler

**Discriminative learning rates:** lower BERT layers learned fundamental legal syntax and need minimal adjustment. The classification head starts random and needs the highest LR.

**Gradual unfreezing:** bottom 6 layers frozen for the first `FREEZE_EPOCHS`. Without this, noisy gradients from the random classification head can corrupt the pre-trained representations in the first epoch.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW([
    # Bottom 6 BERT layers — slowest (fundamental syntax)
    {'params': model.bert.encoder.layer[:6].parameters(),
     'lr': CFG['BASE_LR'] / 2,  'weight_decay': CFG['WEIGHT_DECAY']},
    # Top 6 BERT layers — standard LR (task-specific features)
    {'params': model.bert.encoder.layer[6:].parameters(),
     'lr': CFG['BASE_LR'],      'weight_decay': CFG['WEIGHT_DECAY']},
    # Pooler
    {'params': model.bert.pooler.parameters(),
     'lr': CFG['BASE_LR'],      'weight_decay': CFG['WEIGHT_DECAY']},
    # Classification head — fastest (starts from random init)
    {'params': model.classifier.parameters(),
     'lr': CFG['HEAD_LR'],      'weight_decay': 0.0},
], eps=1e-8)

total_steps  = len(tr_loader) * CFG['EPOCHS'] // CFG['ACCUM_STEPS']
warmup_steps = int(CFG['WARMUP_RATIO'] * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)

print(f'Total optimizer steps: {total_steps}')
print(f'Warmup steps:          {warmup_steps}')
print(f'Effective batch size:  {CFG["BATCH_SIZE"] * CFG["ACCUM_STEPS"]}')

## Cell 8 — Evaluation Helper

Threshold sweep each epoch: finds the threshold in `[THRESH_MIN, THRESH_MAX]` that maximises macro-F1 on the validation set. This replicates the Week 3 ablation B2 result (+0.138 macro-F1 vs default 0.5).

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

@torch.no_grad()
def evaluate(model, loader, threshold_sweep=True):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        all_logits.append(model(ids, mask).cpu())
        all_labels.append(batch['labels'].cpu())

    logits = torch.cat(all_logits).numpy()   # (N, 41)
    labels = torch.cat(all_labels).numpy()   # (N, 41)
    proba  = 1 / (1 + np.exp(-logits))       # sigmoid

    if threshold_sweep:
        best_t, best_m = 0.5, 0.0
        for t in np.arange(CFG['THRESH_MIN'], CFG['THRESH_MAX'], CFG['THRESH_STEP']):
            m = f1_score(labels, (proba >= t).astype(int),
                         average='macro', zero_division=0)
            if m > best_m: best_m, best_t = m, t
        thresh = best_t
    else:
        thresh = 0.5

    preds = (proba >= thresh).astype(int)
    micro = f1_score(labels, preds, average='micro',  zero_division=0)
    macro = f1_score(labels, preds, average='macro',  zero_division=0)
    try:    auroc = roc_auc_score(labels, proba, average='macro')
    except: auroc = float('nan')

    return {
        'micro':     round(float(micro),  4),
        'macro':     round(float(macro),  4),
        'auroc':     round(float(auroc),  4),
        'threshold': round(float(thresh), 2),
        'proba':     proba,
        'labels':    labels,
        'per_class': f1_score(labels, preds, average=None, zero_division=0),
    }

print('evaluate() defined.')

## Cell 9 — Training Loop

This cell runs the full fine-tuning. Watch the `Val MF1` column — it should rise each epoch. Early stopping fires if it doesn't improve for `PATIENCE` consecutive epochs.

In [ ]:
import time, json
from pathlib import Path
from tqdm.notebook import tqdm

best_val_macro = 0.0
best_threshold = CFG['THRESH_MIN']
patience_count = 0
history        = []

print(f'Training for up to {CFG["EPOCHS"]} epochs')
print(f'Early stopping patience: {CFG["PATIENCE"]} epochs')
print(f'Saving to: {CFG["SAVE_DIR"]}')
print('=' * 72)

for epoch in range(CFG['EPOCHS']):
    t0 = time.time()

    # ── Gradual unfreezing ─────────────────────────────────────────────────
    frozen = epoch < CFG['FREEZE_EPOCHS']
    for layer in model.bert.encoder.layer[:6]:
        for p in layer.parameters():
            p.requires_grad = not frozen
    freeze_tag = ' [layers 0-5 frozen]' if frozen else ' [all layers active]'

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(tr_loader, desc=f'Ep {epoch+1}/{CFG["EPOCHS"]}{freeze_tag}',
                leave=True)

    for step, batch in enumerate(pbar):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['labels'].to(DEVICE)

        logits = model(ids, mask)
        loss   = criterion(logits, lbls) / CFG['ACCUM_STEPS']
        loss.backward()
        total_loss += loss.item() * CFG['ACCUM_STEPS']

        if (step + 1) % CFG['ACCUM_STEPS'] == 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), CFG['MAX_GRAD_NORM'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix(loss=f'{total_loss / (step + 1):.4f}')

    avg_loss = total_loss / len(tr_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    vm      = evaluate(model, va_loader, threshold_sweep=True)
    elapsed = int(time.time() - t0)

    print(f'  Ep {epoch+1}  Loss={avg_loss:.4f}  '
          f'Val µF1={vm["micro"]:.4f}  Val MF1={vm["macro"]:.4f}  '
          f'AUROC={vm["auroc"]:.4f}  Thresh={vm["threshold"]:.2f}  '
          f'({elapsed}s)')

    rec = {'epoch': epoch+1, 'loss': round(avg_loss, 4),
           'val_micro': vm['micro'], 'val_macro': vm['macro'],
           'val_auroc': vm['auroc'], 'val_thresh': vm['threshold']}
    history.append(rec)

    # ── Save best ──────────────────────────────────────────────────────────
    if vm['macro'] > best_val_macro:
        best_val_macro = vm['macro']
        best_threshold = vm['threshold']
        patience_count = 0
        torch.save(model.state_dict(),
                   f'{CFG["SAVE_DIR"]}/pytorch_model.bin')
        print(f'  ✓ New best Val MF1={best_val_macro:.4f} '
              f'@ threshold={best_threshold:.2f}  → saved')
    else:
        patience_count += 1
        print(f'  No improvement ({patience_count}/{CFG["PATIENCE"]})')
        if patience_count >= CFG['PATIENCE']:
            print(f'  Early stopping at epoch {epoch+1}')
            break

print(f'\nTraining complete.')
print(f'Best val macro-F1: {best_val_macro:.4f} @ threshold={best_threshold:.2f}')

## Cell 10 — Final Test Evaluation

In [ ]:
# Reload best checkpoint
model.load_state_dict(
    torch.load(f'{CFG["SAVE_DIR"]}/pytorch_model.bin', map_location=DEVICE)
)

# Evaluate at best validation threshold
te_m   = evaluate(model, te_loader, threshold_sweep=False)
preds  = (te_m['proba'] >= best_threshold).astype(int)
te_micro = float(f1_score(te_m['labels'], preds, average='micro',  zero_division=0))
te_macro = float(f1_score(te_m['labels'], preds, average='macro',  zero_division=0))
te_pc    = f1_score(te_m['labels'], preds, average=None, zero_division=0)

print('=' * 65)
print('FINAL TEST RESULTS')
print('=' * 65)
print(f'  µF1  = {te_micro:.4f}')
print(f'  MF1  = {te_macro:.4f}  (threshold = {best_threshold:.2f})')
print(f'  AUROC= {te_m["auroc"]:.4f}')
print()
print('  Baseline comparison (same iterative split, seed=42):')
print(f'    LR (balanced, bigram):  µF1=0.8071  MF1=0.6358  AUROC=0.8574')
print(f'    RF + thresh=0.20:       µF1=0.7318  MF1=0.6533  AUROC=0.8525  ← best classical')
print(f'    MLP (TF-IDF 500-dim):   µF1=0.6168  MF1=0.5119  AUROC=0.8272')
print(f'    LegalBERT (this run):   µF1={te_micro:.4f}  MF1={te_macro:.4f}  AUROC={te_m["auroc"]:.4f}')
print()

# ── The key hypothesis from Week 3 ────────────────────────────────────────────
cap_i   = CLAUSE_TYPES.index('Cap On Liability')
uncap_i = CLAUSE_TYPES.index('Uncapped Liability')
print(f'  KEY HYPOTHESIS CHECK (Week 3 failure mode):')
print(f'    Cap On Liability:    LR=0.733  LegalBERT={te_pc[cap_i]:.3f}')
print(f'    Uncapped Liability:  LR=0.000  LegalBERT={te_pc[uncap_i]:.3f}')
if te_pc[uncap_i] >= 0.60:
    print('    ✓ CONFIRMED: LegalBERT resolves negation polarity confusion')
else:
    print('    → Note: Check AUROC per-label for Uncapped Liability')
    print('      (may have insufficient test positives for reliable F1 estimate)')

print()
print('  Per-clause F1 (ascending):')
print(f'  {"Clause Type":<42} F1')
print('  ' + '-' * 52)
for name, f1 in sorted(zip(CLAUSE_TYPES, te_pc.tolist()), key=lambda x: x[1]):
    bar  = '█' * int(f1 * 20)
    flag = '  ⚠ check AUROC' if f1 == 0.0 else ''
    print(f'  {name:<42} {f1:.3f}  {bar}{flag}')

## Cell 11 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

NAVY, TEAL, AMBER, RED = '#0F2444', '#0D9488', '#B45309', '#B91C1C'
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
epochs_x = [r['epoch'] for r in history]

ax = axes[0]
ax.plot(epochs_x, [r['loss'] for r in history], 'o-', color=NAVY, lw=2.2, label='Train loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCEWithLogitsLoss')
ax.set_title('Training Loss', fontsize=12)
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
ax.plot(epochs_x, [r['val_macro'] for r in history], 's-',
        color=TEAL, lw=2.2, ms=7, label='Val Macro-F1')
ax.plot(epochs_x, [r['val_micro'] for r in history], 'o--',
        color=NAVY, lw=1.8, ms=5, alpha=0.6, label='Val Micro-F1')
best_ep = max(range(len(history)), key=lambda i: history[i]['val_macro'])
ax.axvline(history[best_ep]['epoch'], color=AMBER, ls='--', lw=1.5,
           label=f'Best ep {history[best_ep]["epoch"]}')
# Baseline reference lines
ax.axhline(0.6533, color=RED, ls=':', lw=1.2, alpha=0.7, label='RF+thresh (W3)')
ax.axhline(0.6358, color=NAVY, ls=':', lw=1.0, alpha=0.5, label='LR baseline (W2)')
ax.set_xlabel('Epoch'); ax.set_ylabel('F1')
ax.set_title('Validation F1 vs Classical Baselines', fontsize=12)
ax.legend(fontsize=8); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[2]
models = ['LR\nbaseline', 'RF+\nthreshold', 'MLP\nbest', 'LegalBERT\n(this run)']
micros = [0.8071, 0.7318, 0.6168, te_micro]
macros = [0.6358, 0.6533, 0.5119, te_macro]
x = np.arange(4); w = 0.35
b1 = ax.bar(x-w/2, micros, w,
             color=[NAVY, '#1D6FA4', '#0D948866', TEAL], alpha=0.9, label='µF1')
b2 = ax.bar(x+w/2, macros, w,
             color=[NAVY+'88', '#1D6FA488', '#0D948844', TEAL+'CC'], alpha=0.9, label='MF1')
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
            ha='center', va='bottom', fontsize=7, rotation=90)
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=8.5)
ax.set_ylabel('F1'); ax.set_ylim(0, 1.12)
ax.set_title('All Models — Weeks 2→6\n(same iterative split)', fontsize=12)
ax.legend(fontsize=9)
ax.axhline(0.80, color=RED, ls=':', lw=1, alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_path = f'{CFG["SAVE_DIR"]}/training_curves.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Cell 12 — Save Everything to Drive

In [ ]:
# Save config + metrics + history
output = {
    'config': CFG,
    'training_history': history,
    'best_val_macro': best_val_macro,
    'best_threshold': best_threshold,
    'test_micro_f1': te_micro,
    'test_macro_f1': te_macro,
    'test_auroc':    te_m['auroc'],
    'test_per_clause_f1': dict(zip(CLAUSE_TYPES, te_pc.tolist())),
    'clause_types': CLAUSE_TYPES,
    'model_name': CFG['MODEL_NAME'],
}

results_path = f'{CFG["SAVE_DIR"]}/results.json'
with open(results_path, 'w') as f:
    json.dump(output, f, indent=2)

print('Saved to Google Drive:')
print(f'  {CFG["SAVE_DIR"]}/pytorch_model.bin     — model weights')
print(f'  {CFG["SAVE_DIR"]}/results.json           — all metrics + config')
print(f'  {CFG["SAVE_DIR"]}/training_curves.png    — figures')

print(f'\nFINAL SUMMARY')
print(f'  Test µF1:  {te_micro:.4f}')
print(f'  Test MF1:  {te_macro:.4f}  (threshold={best_threshold:.2f})')
print(f'  Test AUROC: {te_m["auroc"]:.4f}')
print(f'  Cap F1:     {te_pc[cap_i]:.3f}  (LR was 0.733)')
print(f'  Uncap F1:   {te_pc[uncap_i]:.3f}  (LR was 0.000)')

print('\nTo use the saved model:')
print("""
  # In your app or demo:
  from transformers import AutoTokenizer, AutoModel
  import torch
  import torch.nn as nn

  class LegalBERTClassifier(nn.Module):
      def __init__(self, model_name, num_labels=41):
          super().__init__()
          self.bert       = AutoModel.from_pretrained(model_name)
          self.dropout    = nn.Dropout(0.1)
          self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
      def forward(self, input_ids, attention_mask, **kwargs):
          out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
          return self.classifier(self.dropout(out.last_hidden_state[:, 0, :]))

  model = LegalBERTClassifier('nlpaueb/bert-base-uncased-contracts')
  model.load_state_dict(torch.load('pytorch_model.bin', map_location='cpu'))
  model.eval()

  tokenizer = AutoTokenizer.from_pretrained('nlpaueb/bert-base-uncased-contracts')
""")

## Cell 13 — Sliding-Window Inference (full contracts)

Run this on any contract text after training. Handles contracts longer than 512 tokens via max-pooled sliding window.

In [ ]:
CLAUSE_RISK_WEIGHT = {
    'Uncapped Liability': 10, 'Limitation Of Liability': 9, 'Cap On Liability': 9,
    'IP Ownership Assignment': 9, 'Joint IP Ownership': 8, 'Change Of Control': 8,
    'Anti-Assignment': 7, 'Liquidated Damages': 7, 'Source Code Escrow': 7,
    'Unlimited License': 6, 'Non-Compete': 6, 'Audit Rights': 6,
    'Unilateral Termination': 6, 'Covenant Not To Sue': 6, 'Most Favored Nation': 6,
}

@torch.no_grad()
def score_contract(text, threshold=None):
    """
    Full pipeline: text → risk scorecard.
    Uses max-pool sliding window to handle contracts > 512 tokens.
    """
    thresh = threshold or best_threshold
    model.eval()

    full     = tokenizer(text, add_special_tokens=False,
                         return_tensors='pt')['input_ids'][0]
    cls_id   = tokenizer.cls_token_id
    sep_id   = tokenizer.sep_token_id
    pad_id   = tokenizer.pad_token_id or 0
    content  = CFG['MAX_LENGTH'] - 2
    cls_embs = []

    for start in range(0, len(full), CFG['STRIDE']):
        end    = min(start + content, len(full))
        chunk  = full[start:end]
        padded = torch.full((CFG['MAX_LENGTH'],), pad_id, dtype=torch.long)
        padded[0] = cls_id
        padded[1:len(chunk)+1] = chunk
        padded[len(chunk)+1]   = sep_id
        mask = torch.zeros(CFG['MAX_LENGTH'], dtype=torch.long)
        mask[:len(chunk)+2] = 1

        out = model.bert(input_ids=padded.unsqueeze(0).to(DEVICE),
                         attention_mask=mask.unsqueeze(0).to(DEVICE))
        cls_embs.append(out.last_hidden_state[0, 0, :])
        if end == len(full): break

    pooled = torch.stack(cls_embs).max(dim=0).values
    logits = model.classifier(model.dropout(pooled.unsqueeze(0)))[0]
    proba  = torch.sigmoid(logits).cpu().numpy()
    flags  = (proba >= thresh).astype(int)

    print(f'Clauses detected (threshold={thresh:.2f}):')
    detected = [(CLAUSE_TYPES[i], float(proba[i])) for i in range(41) if flags[i]]
    detected.sort(key=lambda x: -x[1])
    for name, p in detected:
        risk = CLAUSE_RISK_WEIGHT.get(name, 3)
        bar  = '█' * int(p * 20)
        flag = ' ⚠ HIGH RISK' if risk >= 8 else ' ⚠' if risk >= 6 else ''
        print(f'  {name:<42} {p:.2f}  {bar}{flag}')

    print(f'\nTotal detected: {len(detected)}/41')
    return proba

# Test on a sample
SAMPLE = """This Software as a Service Agreement grants Customer a non-exclusive,
non-transferable license. All intellectual property ownership shall be assigned
to Vendor. There shall be no cap on liability and uncapped liability applies to
all indemnification. The agreement auto-renews annually with 90-day notice period
to terminate renewal. Liquidated damages of $500,000 apply for breach of exclusivity.
Change of control requires Vendor consent. Source code held in escrow."""

print('=== Sample contract inference ===')
_ = score_contract(SAMPLE)